In [1]:
import pandas as pd

df = pd.read_csv(
    "train-marked.csv",
    dtype={
        "file": "string",
        "speaker": "string",
        "turn": "int64",
        "text_fluent": "string",
        "text_disfluent": "string",
        "keep": "boolean",   # pandas nullable boolean
    }
)

df = df[df["keep"]==True]

print("# of samples:", len(df))
# display(df)

# of samples: 80


In [2]:
import pandas as pd
import re
from pathlib import Path
from difflib import SequenceMatcher
from IPython.display import display

# -------------------------
# Paths
# -------------------------
DATA_DIR = Path(".")
TS_DIR = DATA_DIR / "swb1_audio_timestamps"
TRAIN_FILE = DATA_DIR / "train-marked.csv"
OUT_FILE = DATA_DIR / "translation-dataset-with-timestamps.csv"


# -------------------------
# Normalization
# -------------------------
def norm_token(tok):
    tok = tok.lower()
    tok = re.sub(r"\[.*?\]", "", tok)   # [silence], [noise], etc
    tok = tok.replace("-", "")          # repairs: h[ow]- → how
    return tok.strip()


def tokenize_disfluent(text):
    if pd.isna(text):
        return []
    text = text.replace("_", " ").lower()
    return re.findall(r"[a-z']+", text)


# -------------------------
# Load word timestamps
# -------------------------
def load_word_ts(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 4:
                continue
            _, start, end, word = parts[:4]
            w = norm_token(word)
            if not w:
                continue
            rows.append({
                "word": w,
                "start": float(start),
                "end": float(end),
            })
    return pd.DataFrame(rows).reset_index(drop=True)


# -------------------------
# Load train data
# -------------------------
train_df = pd.read_csv(TRAIN_FILE)

start_time = [None] * len(train_df)
end_time = [None] * len(train_df)


# -------------------------
# Find all conversations
# -------------------------
# sw2005A-ms98-a-word.text → sw2005
conv_ids = sorted({
    f.name.split("A-")[0]
    for f in TS_DIR.glob("*A-ms98-a-word.text")
})


# -------------------------
# Process each conversation
# -------------------------
for conv in conv_ids:
    print(f"Processing {conv}")

    for speaker in ("A", "B"):
        word_file = TS_DIR / f"{conv}{speaker}-ms98-a-word.text"
        if not word_file.exists():
            continue

        ts_df = load_word_ts(word_file)
        ts_tokens = ts_df["word"].tolist()

        # rows belonging to this conversation + speaker
        mask = (train_df["file"] == conv) & (train_df["speaker"] == speaker)
        rows = train_df[mask]

        if rows.empty or ts_df.empty:
            continue

        # flatten train tokens
        train_tokens = []
        token_to_row = []

        for idx, row in rows.iterrows():
            toks = tokenize_disfluent(row["text_disfluent"])
            for t in toks:
                train_tokens.append(t)
                token_to_row.append(idx)

        if not train_tokens:
            continue

        # global monotonic alignment
        sm = SequenceMatcher(
            None,
            train_tokens,
            ts_tokens,
            autojunk=False
        )

        row_to_ts_idxs = {}

        for tag, i1, i2, j1, j2 in sm.get_opcodes():
            if tag in ("equal", "replace"):
                n = min(i2 - i1, j2 - j1)
                for k in range(n):
                    row_idx = token_to_row[i1 + k]
                    ts_idx = j1 + k
                    row_to_ts_idxs.setdefault(row_idx, []).append(ts_idx)

        # assign spans
        for row_idx, ts_idxs in row_to_ts_idxs.items():
            start_time[row_idx] = ts_df.loc[min(ts_idxs), "start"]
            end_time[row_idx] = ts_df.loc[max(ts_idxs), "end"]


# -------------------------
# Attach + save
# -------------------------
merged = train_df.copy()
merged["start_time"] = start_time
merged["end_time"] = end_time

# filter to only keep the rows in our translation dataset
merged = merged[merged["keep"] == True]
merged = merged.drop(columns=["keep"])
display(merged)

merged.to_csv(OUT_FILE, index=False)

print(f"\nSaved: {OUT_FILE}")
print("Total unaligned rows:", merged["start_time"].isna().sum())

Processing sw2005
Processing sw2008
Processing sw2010
Processing sw2012
Processing sw2015
Processing sw2018
Processing sw2020
Processing sw2022
Processing sw2024
Processing sw2027
Processing sw2028
Processing sw2032


,file,speaker,turn,text_fluent,text_disfluent,start_time,end_time
1,sw2005,B,2,"of course, it's one of the last few things in ...","_Well_, of course, _it_ _'s_ _, _ _you_ _know_...",11.233375,21.142500
3,sw2005,B,4,"I'd be very very careful checking them out, ou...","I'd be very very careful _and_ _, _ _uh_ _, _ ...",22.219000,34.899000
7,sw2005,B,8,"So, I was very comfortable in doing it when it...","So, I was very comfortable _, _ _you_ _know_ _...",45.737750,75.572750
13,sw2005,B,14,is there something else we could have done in ...,"_You_ _know_ _, _ is there something else we c...",88.623375,98.993375
18,sw2005,A,19,"Probably the hardest thing in my family, my gr...",_Uh-huh_ _. _ _Yeah_ _. _ Probably the hardest...,99.452375,117.519500
...,...,...,...,...,...,...,...
1105,sw2028,A,125,"taking care of, I'm actually in the air divisi...","_Taking_ _, _ taking care of _uh_, I'm actuall...",472.294250,501.071250
1127,sw2028,A,147,"And, our chemical data base, so that we know e...","And, _uh_, _our_ _, _ our chemical data base, ...",580.838375,599.556250
1166,sw2032,A,39,"But, for example, they have, what, but other b...","But, _see_, for example, they have, _see_ what...",102.677875,115.740125
1204,sw2032,A,77,"they're still around, they've got a new C D ou...","_Yeah_ _, _ _they_ _'re_ _, _ they're still ar...",216.561500,227.356375



Saved: translation-dataset-with-timestamps.csv
Total unaligned rows: 1


In [3]:
!pwd

/Users/mariateleki/Desktop/Uh-Mazing/data


In [2]:
import pandas as pd
import subprocess
from pathlib import Path

# -------------------------
# Paths
# -------------------------
CSV_FILE = Path("translation-dataset-with-timestamps.csv")
AUDIO_DIR = Path("swb1_LDC97S62_wav")
OUT_DIR = Path("translation_dataset_audio")

OUT_DIR.mkdir(parents=True, exist_ok=True)

BUFFER = 2.0  # seconds

# -------------------------
# Load data
# -------------------------
df = pd.read_csv(CSV_FILE)
df = df.dropna(subset=["start_time", "end_time"])

print(f"Exporting {len(df)} audio segments")

# -------------------------
# Helper: get wav duration
# -------------------------
def wav_duration(path: Path) -> float:
    cmd = [
        "ffprobe",
        "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(path),
    ]
    out = subprocess.check_output(cmd).decode().strip()
    return float(out)

# cache durations (one ffprobe per wav)
dur_cache = {}

# -------------------------
# Export audio
# -------------------------
for _, row in df.iterrows():
    conv_raw = row["file"]          # "sw2005" or possibly "2005"
    conv_num = int(conv_raw.replace("sw", ""))
    conv = f"sw{conv_num:05d}"      # -> "sw02005"

    in_wav = AUDIO_DIR / f"{conv}.wav"

    if not in_wav.exists():
        print(f"Missing audio: {in_wav}")
        continue

    if conv not in dur_cache:
        dur_cache[conv] = wav_duration(in_wav)

    wav_dur = dur_cache[conv]

    start = max(0.0, float(row["start_time"]) - BUFFER)
    end = min(wav_dur, float(row["end_time"]) + BUFFER)
    dur = max(0.0, end - start)
    turn = row["turn"]
    speaker = row["speaker"]

    if dur <= 0:
        continue

    out_wav = OUT_DIR / f"{conv}_{speaker}_{turn}.wav"

    cmd = [
        "ffmpeg",
        "-y",
        "-loglevel", "error",
        "-ss", f"{start:.3f}",
        "-t", f"{dur:.3f}",
        "-i", str(in_wav),
        "-acodec", "pcm_s16le",
        "-ar", "16000",
        str(out_wav),
    ]

    subprocess.run(cmd, check=False)

print("Done.")


Exporting 79 audio segments
Done.
